### Global Electronics Retailer
Sales data for a fictitious global electronics retailer, including tables containing information about transactions, products, customers, stores and currency exchange rates.

### BUSINESS QUESTIONS
What types of products does the company sell, and where are customers located?

Is there a difference in average order value (AOV) for online vs. in-store sales?

How long is the average delivery time in days? Has that changed over time?

Are there any seasonal patterns or trends for order volume or revenue?


In [0]:
%sql
USE electronics.global_electronics;

 What types of products does the company sell?


### Product Portfolio Analysis**
Based on the data, the company sells 2,517 products across 8 main categories in the electronics sector:

### Key Findings:
Largest Categories:**

Home Appliances (661 products, 26%) - The dominant category
Computers (606 products, 24%) - Second largest offering
Cameras and camcorders (372 products, 15%)
Mid-Size Categories:

Cell phones (285 products, 11%)
TV and Video (222 products, 9%)
Games and Toys (166 products, 7%)
Smaller Categories:

Audio (115 products, 5%)
Music, Movies and Audio Books (90 products, 4%)

In [0]:
%sql
SELECT 
    Category
   -- ,Subcategory 
    ,count(*) As Count_of_Products
FROM products P
GROUP BY 
    P.Category
  --  ,P.Subcategory
    ;


Databricks visualization. Run in Databricks to view.

# Where are customers located?

US-Centric Business: The United States alone represents 45% of the entire customer base, making it the dominant market requiring focused attention

**Continental Split:**

North America: 55% (8,381 customers)
Europe: 36% (6,465 customers)
Australia/Pacific: 9% (1,420 customers)

In [0]:
%sql
SELECT 
         Continent
        ,Country
      --  ,State 
     --   ,CONCAT(City, ' ',`State Code`) AS city
        ,COUNT(*) AS Number_Of_Customers
        ,ROUND(
            COUNT(*)/SUM(COUNT(*)) OVER() 
        ,2) AS Percentage
FROM customers C
GROUP BY 
        Continent
        ,Country
     --   ,State 
     --   ,CONCAT(City, ' ',`State Code`)
ORDER BY Number_Of_Customers DESC;

Databricks visualization. Run in Databricks to view.

Are there any seasonal patterns or trends for order volume?

# Are there any seasonal patterns or trends by Order Volume?


Winter Holiday Effect: Winter consistently drives 35-40% (78,043 orders) of annual orders, indicating strong holiday shopping season dependency

Spring Opportunity: Spring represents only 14% of orders - potential opportunity for promotional campaigns to boost this historically weak quarter

Business Disruption: The 2020 pandemic caused a 50% volume drop, with recovery pattern unclear from available data

Seasonality Ratio: Winter orders are 2.8x higher than Spring orders on average, showing extreme seasonal concentration

In [0]:
%sql
WITH order_volumn_Season_trend AS(
SELECT 
    YEAR(`Order Date`) AS Year,
    CASE 
        WHEN MONTH(`Order Date`) BETWEEN 3 AND 5 THEN 'Spring'
        WHEN MONTH(`Order Date`) BETWEEN 6 AND 8 THEN 'Summer'
        WHEN MONTH(`Order Date`) BETWEEN 9 AND 11 THEN 'Autumn'
        ELSE 'Winter'
    END AS Season
    ,SUM(Quantity) As Number_Of_Orders
FROM sales
GROUP BY Year, Season
)
SELECT 
    Year
    ,Season
   ,Number_Of_Orders
FROM order_volumn_Season_trend;

Databricks visualization. Run in Databricks to view.

## Are there any seasonal patterns or trends for revenue?

Winter generates 3.4x more revenue than Spring, creating significant cash flow concentration. 

The pandemic halved revenue while maintaining AOV, confirming the drop was purely demand-side (fewer customers ordering), not value-side (lower spending per order).

With only 12% of annual revenue, Spring represents the biggest untapped potential for promotional campaigns to smooth revenue throughout the year

In [0]:
%sql
SELECT 
YEAR(`Order Date`) AS Year,
        CASE 
            WHEN MONTH(`Order Date`) BETWEEN 3 AND 5 THEN 'Spring'
            WHEN MONTH(`Order Date`) BETWEEN 6 AND 8 THEN 'Summer'
            WHEN MONTH(`Order Date`) BETWEEN 9 AND 11 THEN 'Autumn'
            ELSE 'Winter'
        END AS Season
        ,SUM(Quantity) As Number_Of_Orders
        ,ROUND(SUM(Quantity * Unit_Price_USD),2) As Total_Revenue
        ,ROUND(AVG(Quantity * Unit_Price_USD),2) As AVG_Revenue
FROM sales s
INNER JOIN products p
GROUP BY Year, Season
ORDER BY Year;

Databricks visualization. Run in Databricks to view.

**Is there a difference in average order value (AOV) for online vs. in-store sales?**

- In-store sales have a slightly higher AOV than online sales, with customers spending approximately $283.49 per in-store order compared with $276.06 online. This represents a 2.7% difference, suggesting that customer spending per order is broadly similar across both channels.

In [0]:
%sql

SELECT 
    CASE WHEN s.Storekey >= 1 Then 'Instore' Else 'Online' END AS Store_Type
   -- ,SUM(s1.Quantity) AS Quantity_Sold
    -- ,ROUND(AVG(Quantity * Unit_Price_USD),2) As AVG_Revenue
    ,ROUND(SUM(Quantity * Unit_Price_USD)/SUM(s1.Quantity),2) AS AOV
FROM stores s
INNER JOIN sales s1
ON s.StoreKey = s1.StoreKey
INNER JOIN products p
ON s1.ProductKey = p.ProductKey
GROUP BY Store_Type;  

Databricks visualization. Run in Databricks to view.

*** How long is the average delivery time in days? Has that changed over time?

Delivery Time Analysis
Average delivery time: 4 days (ranging from 1 to 17 days)

Yes, there's been significant improvement over time:

2016: Started at 4.6 days average
2017-2018: Steady decline from 4.32 → 4.03 days
2019-2020: Stabilized around 4.0 days
2021 Q1: Best performance at 3.96 days
The company achieved a 14% improvement in delivery speed (4.6 → 3.96 days) from early 2016 to 2021, with most gains realized by 2018. 

In [0]:
%sql
SELECT 
    YEAR(`Order Date`) AS Year,
    QUARTER(`Order Date`) AS Quarter,
    ROUND(AVG(DATEDIFF(`Delivery Date`, `Order Date`)) ,2) AS Avg_Delivery_Days,
    COUNT(*) AS Order_Count
FROM sales
WHERE `Delivery Date` IS NOT NULL
GROUP BY Year, Quarter
ORDER BY Year, Quarter;

Databricks visualization. Run in Databricks to view.

How long is the avearge delivery time in days

Average delivery time: 4 days

The delivery times range from 1 day (fastest) to 17 days (slowest), with most orders delivered around the 4-day average.

In [0]:
%sql

SELECT 
    MIN(DATEDIFF(`Delivery Date`,`Order Date`)) AS `Mininum Days`
    ,ROUND(AVG(DATEDIFF(`Delivery Date`,`Order Date`)),0) AS `Average Days`
    ,MAX(DATEDIFF(`Delivery Date`,`Order Date`)) AS `Maximum Days`
FROM sales
WHERE `Delivery Date` IS NOT NULL;

Databricks visualization. Run in Databricks to view.